In [ ]:
import json
import pandas as pd
from datasets import Dataset
from transformers import MT5Tokenizer

DATA_PATH = "../data/processed/dataset.json"
MODEL_NAME = "google/mt5-small"

In [ ]:
with open(DATA_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

print("Taille dataset:", len(data))
data[:3]

In [ ]:
rows = []

for item in data:
    rows.append({
        "hausa": item["translation"]["hausa"],
        "zarma": item["translation"]["zarma"]
    })

df = pd.DataFrame(rows)

df.head()

In [ ]:
df["hausa"] = df["hausa"].str.strip()
df["zarma"] = df["zarma"].str.strip()

# enlever lignes vides
df = df[(df["hausa"] != "") & (df["zarma"] != "")]

In [ ]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

In [ ]:
train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
val_dataset = Dataset.from_pandas(val_df.reset_index(drop=True))
test_dataset = Dataset.from_pandas(test_df.reset_index(drop=True))

In [ ]:
tokenizer = MT5Tokenizer.from_pretrained(MODEL_NAME)

In [ ]:
PREFIX = "translate Hausa to Zarma: "MAX_INPUT_LENGTH = 32
MAX_TARGET_LENGTH = 32

def preprocess_function(examples):
    inputs = [PREFIX + text for text in examples["hausa"]]
    targets = examples["zarma"]

    model_inputs = tokenizer(
        inputs,
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
        padding="max_length"
    )

    labels = tokenizer(
        targets,
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
        padding="max_length"
    )

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

In [ ]:
tokenized_train = train_dataset.map(preprocess_function, batched=True)
tokenized_val = val_dataset.map(preprocess_function, batched=True)
tokenized_test = test_dataset.map(preprocess_function, batched=True)

In [ ]:
tokenized_train[0]

In [ ]:
sample = tokenized_train[0]

decoded_input = tokenizer.decode(sample["input_ids"], skip_special_tokens=True)
decoded_label = tokenizer.decode(sample["labels"], skip_special_tokens=True)

print("INPUT :", decoded_input)
print("TARGET:", decoded_label)

In [ ]:
tokenized_train.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
tokenized_val.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
tokenized_test.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

In [ ]:
tokenized_train.save_to_disk("../data/processed/train_dataset")
tokenized_val.save_to_disk("../data/processed/val_dataset")
tokenized_test.save_to_disk("../data/processed/test_dataset")

"""
Observations:

1. Données converties en format compatible Transformer
2. Utilisation du préfixe pour guider mT5
3. Longueur max fixée à 32 (adaptée aux phrases courtes)
4. Padding appliqué pour batch training
5. Labels correctement alignés avec targets

Conclusion:
Dataset prêt pour fine-tuning.
"""